<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReAct AI Agent Loop Demo

This notebook demonstrates a **ReAct (Reason + Act) agent** using the Gemini API and a document-search tool.

The agent can:
1. Receive a user question.
2. Decide whether it needs to search the supplied documents.
3. Call the `search_documents` tool.
4. Observe the tool result.
5. Continue and produce a final answer.

Complete the cells from top to bottom. The document collection below contains four Computer Science documents, each with at least four sentences.

## Setup — Gemini API key

1. Create a Gemini API key at https://aistudio.google.com/apikey
2. In Google Colab, open the **Secrets** panel (key icon).
3. Add a secret named `GOOGLE_API_KEY`, paste your key as its value, and enable notebook access.
4. Run the API-key cell below.

**Do not write your API key directly in a code cell or commit it to GitHub.**

## References
- Gemini API models: https://ai.google.dev/gemini-api/docs/models
- Gemini API function calling: https://ai.google.dev/gemini-api/docs/function-calling
- Google Gen AI Python SDK: https://googleapis.github.io/python-genai/


In [1]:
import os
import getpass

# First try to read the key securely from Google Colab Secrets.
api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    # This path is also useful when running outside Google Colab.
    api_key = None

# Ask privately if the Colab Secret was not configured.
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key (input is hidden): ")

if not api_key or not api_key.strip():
    raise ValueError("A Gemini API key is required.")

os.environ["GOOGLE_API_KEY"] = api_key.strip()
del api_key

print("Gemini API key is configured.")


Gemini API key is configured.


In [3]:
# Install the current Google Gen AI Python SDK.
%pip install -q -U google-genai
print("Google Gen AI SDK installed.")


Google Gen AI SDK installed.


In [4]:
# @title User input needed — document collection

# Each document must contain at least four sentences.
# You may replace these Computer Science topics with another topic of your choice.

DOCUMENTS = {
    "stacks": (
        "A stack is a linear data structure that follows the Last In First Out principle. "
        "The two main operations are push and pop. "
        "Push inserts an element at the top of the stack, while pop removes the top element. "
        "Peek returns the top element without removing it. "
        "Stacks are used in function calls, undo operations, and expression evaluation."
    ),

    "binary_search": (
        "Binary search is an efficient searching algorithm used on sorted arrays. "
        "It compares the target value with the middle element of the array. "
        "If the target is smaller, the search continues in the left half. "
        "If the target is larger, the search continues in the right half. "
        "Its time complexity is O(log n), and iterative binary search uses O(1) extra space."
    ),

    "java_oop": (
        "Object Oriented Programming in Java is based on classes and objects. "
        "Encapsulation combines data and methods inside a class. "
        "Inheritance allows one class to acquire properties and methods of another class. "
        "Polymorphism allows the same interface or method name to have different behaviours. "
        "Abstraction hides implementation details and exposes essential functionality."
    ),

    "react_agents": (
        "A ReAct agent combines reasoning and actions to solve a task. "
        "The agent first decides whether it needs additional information. "
        "It can call a search tool to retrieve relevant information from documents. "
        "The tool returns an observation that the agent can use in its next step. "
        "The agent repeats this process until it can provide a final answer."
    )
}

# Validate the assignment's minimum document requirements.
assert len(DOCUMENTS) >= 4, "Add at least four documents."

for name, content in DOCUMENTS.items():
    sentence_count = len([
        sentence for sentence in content.split(".")
        if sentence.strip()
    ])
    assert sentence_count >= 4, f"{name} must contain at least four sentences."

print(f"Loaded {len(DOCUMENTS)} documents.")
for name in DOCUMENTS:
    print("-", name)


Loaded 4 documents.
- stacks
- binary_search
- java_oop
- react_agents


In [5]:
# @title Define and test the search tool

import re

def search_documents(query: str) -> str:
    """Return the two document passages with the most query-term matches."""

    # Tokenize the query and remove common words that add little search value.
    stop_words = {
        "a", "an", "the", "is", "are", "was", "were", "what", "which",
        "who", "how", "why", "when", "where", "of", "to", "in", "on",
        "for", "and", "or", "with", "about", "explain", "tell", "me",
        "does", "do", "it", "its", "this", "that"
    }

    query_terms = {
        term for term in re.findall(r"[a-z0-9]+", query.lower())
        if term not in stop_words
    }

    if not query_terms:
        return "Please provide a more specific search query."

    results = []

    for doc_name, content in DOCUMENTS.items():
        content_terms = set(re.findall(r"[a-z0-9]+", content.lower()))
        match_count = len(query_terms & content_terms)

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(key=lambda item: item[0], reverse=True)

    if not results:
        return "No relevant documents found for that query."

    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Sanity check — use the function name defined above.
print(search_documents("stack push pop"))


[Source: stacks]
A stack is a linear data structure that follows the Last In First Out principle. The two main operations are push and pop. Push inserts an element at the top of the stack, while pop removes the top element. Peek returns the top element without removing it. Stacks are used in function calls, undo operations, and expression evaluation.


In [11]:
# @title Configure Gemini and the search tool

import time
from google import genai
from google.genai import types

# Explicitly provide the key configured in the API-key cell.
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# Use a current stable Flash model suitable for tool calling.
MODEL_NAME = "gemini-3.5-flash-lite"

search_declaration = types.FunctionDeclaration(
    name="search_documents",
    description=(
        "Search the supplied Computer Science document collection. "
        "Use this tool when a question asks about information that may be "
        "contained in the supplied documents."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The keywords or question to search for."
            }
        },
        "required": ["query"]
    }
)

search_tool = types.Tool(
    function_declarations=[search_declaration]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    system_instruction=(
        "You are a helpful Computer Science learning assistant. "
        "For questions about the supplied document collection, use the "
        "search_documents tool before answering. Base document-specific "
        "answers on the retrieved passages and mention their source names. "
        "If the search tool finds no relevant information, clearly say that "
        "the supplied documents do not contain the answer. "
        "For a clearly general-knowledge question unrelated to the supplied "
        "documents, you may answer from general knowledge, and make clear "
        "that it is not based on the document collection. "
        "Do not invent facts or claim to have searched when you did not."
    )
)

print("Gemini model ready:", MODEL_NAME)


Gemini model ready: gemini-3.5-flash-lite


In [13]:
# @title Run the ReAct agent loop


def send_with_retry(contents, max_retries=4):
    """Retry Gemini requests after temporary API errors."""

    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as error:
            error_text = str(error)

            retryable = any(
                code in error_text
                for code in [
                    "429", "500", "502", "503", "504",
                    "UNAVAILABLE", "RESOURCE_EXHAUSTED"
                ]
            )

            if retryable and attempt < max_retries - 1:
                wait_seconds = 10 * (2 ** attempt)

                print(
                    f"Temporary Gemini error. "
                    f"Retrying in {wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)

            else:
                raise



def run_agent(question: str, max_steps: int = 8) -> str:
    """Run Reason -> Act -> Observe cycles until Gemini returns a final answer."""

    print(f"Question: {question}\n" + "-" * 60)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        if not response.candidates:
            print("Gemini returned no candidate response.")
            return "No answer was returned."

        model_content = response.candidates[0].content

        # Preserve the complete model turn, including any tool-call metadata.
        contents.append(model_content)

        function_calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call is not None
        ]

        # No function call means Gemini has produced its final response.
        if not function_calls:
            final_answer = response.text or "No text answer was returned."
            print(f"\nFinal answer:\n{final_answer}")
            return final_answer

        # Execute each requested tool call and send all observations together.
        function_response_parts = []

        for function_call in function_calls:
            tool_name = function_call.name
            tool_args = dict(function_call.args)

            print(f"\nStep {step}: ACT -> {tool_name}({tool_args})")

            if tool_name == "search_documents":
                try:
                    observation = search_documents(**tool_args)
                except Exception as error:
                    observation = f"Search tool error: {error}"
            else:
                observation = f"Unknown tool: {tool_name}"

            print(f"OBSERVATION -> {observation[:300]}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={"result": observation}
                )
            )

        contents.append(
        types.Content(
        role="user",
        parts=function_response_parts
    )
)

    print("Agent stopped because it reached the maximum number of steps.")
    return "Stopped: maximum agent steps reached."


# First test: this question should invoke the search_documents tool.
_ = run_agent("What are the main operations of a stack?")


Question: What are the main operations of a stack?
------------------------------------------------------------

Step 1: ACT -> search_documents({'query': 'stack operations main push pop'})
OBSERVATION -> [Source: stacks]
A stack is a linear data structure that follows the Last In First Out principle. The two main operations are push and pop. Push inserts an element at the top of the stack, while pop removes the top element. Peek returns the top element without removing it. Stacks are used in functio

Final answer:
Based on the provided document **stacks**, the two main operations of a stack are:

1. **Push**: Inserts an element at the top of the stack.
2. **Pop**: Removes the top element from the stack.

Additionally, another common operation mentioned is **Peek**, which returns the top element without removing it.


In [14]:
# @title Ask follow-up questions

# Include questions answered by the documents, a question not covered
# by the documents, and one general-knowledge question.
questions = [
    "What are the main operations of a stack?",
    "Explain binary search and its time complexity.",
    "What are the four main principles of Object Oriented Programming in Java?",
    "How does a ReAct agent use observations?",
    # This question is not answered by the supplied document collection.
    "How many ghosts eat carrots every day?",
    # This question tests general knowledge rather than the supplied documents.
    "What is a popular food item in Canada?"
]

for i, question in enumerate(questions):
    if i > 0:
        time.sleep(15)  # Helps avoid sending requests too quickly.

    print("\n" + "=" * 60)
    run_agent(question)



Question: What are the main operations of a stack?
------------------------------------------------------------
Temporary Gemini error. Retrying in 10 seconds...

Step 1: ACT -> search_documents({'query': 'stack operations push pop'})
OBSERVATION -> [Source: stacks]
A stack is a linear data structure that follows the Last In First Out principle. The two main operations are push and pop. Push inserts an element at the top of the stack, while pop removes the top element. Peek returns the top element without removing it. Stacks are used in functio

Final answer:
Based on the provided documents, the two main operations of a stack are:

1. **Push**: Inserts an element at the top of the stack.
2. **Pop**: Removes the top element from the stack.

*(Note: Another common operation mentioned is **Peek**, which returns the top element without removing it.)*

**Source:**
- `stacks`

Question: Explain binary search and its time complexity.
----------------------------------------------------------

In [15]:
# Final assignment checks
assert len(DOCUMENTS) >= 4, "You need at least four documents."
assert len(questions) >= 4, "You need at least four questions."
assert all(
    len([s for s in text.split(".") if s.strip()]) >= 4
    for text in DOCUMENTS.values()
), "Every document must contain at least four sentences."

print("All notebook assignment checks passed.")


All notebook assignment checks passed.
